# Интерпретация

## Задание 1. 1 балл
Сделайте 2 версии данных - с нормализацией признаков и без.
Обучите 6 моделей:
- линейную регрессию (LinearRegression) на двух вариантах данных
- Lasso регрессию (Lasso) на двух вариантах данных
- градиентный бустинг (GradientBoostingRegressor) на двух вариантах данных. Ограничьте глубину до 5.

Выведите MSE,RMSE и MAPE моделей. Какая функция больше подходит? Почему?

Зафиксируйте выводы. Какие модели чувствительны к масштабу признаков, а какие почти инвариантны? Почему это важно для анализа признаков?

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'

# Загрузка данных
data = pd.read_csv("data_unreg.csv")

num_features = [
    "location_marketplace_cnt",
    "location_amenity_pharmacy_w_mean_distance",
    "location_public_transport_stop_position_cnt",
    "location_college_cnt",
    "location_railway_cnt",
    "location_logs_count_std",
    "location_leisure_cnt",
    "location_shop_product_w_mean_distance",
    "location_public_transport_platform_w_mean_distance",
    "location_leisure_w_mean_distance",
    "location_flash_mean_mean",
    "location_market_cnt",
    "location_barrier_w_mean_distance",
]
cat_features = [
    "region_name_cat", "class_cat", "district_cat",
    "hc_name_cat", "developer_cat", "corpus_cat", "stage_cat",
]

X = data[num_features + cat_features]
y = data["price_target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
num_none   = Pipeline([("imputer", SimpleImputer(strategy="mean"))])
num_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler()),
])

pre_none   = ColumnTransformer([("num", num_none,   num_features), ("cat", cat_transformer, cat_features)])
pre_scaled = ColumnTransformer([("num", num_scaled, num_features), ("cat", cat_transformer, cat_features)])

def evaluate_model(model, preprocessor):
    pipe = Pipeline([("preprocessor", preprocessor), ("regressor", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    mse   = mean_squared_error(y_test, preds)
    return mse, mse**0.5, mean_absolute_percentage_error(y_test, preds)

models = {
    "LinearRegression": LinearRegression(),
    "Lasso":            Lasso(alpha=0.01, max_iter=10000),
    "GradientBoosting": GradientBoostingRegressor(max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    mse, rmse, mape = evaluate_model(model, pre_none)
    results[f"{name}_no_scaling"] = [mse, rmse, mape]
    mse, rmse, mape = evaluate_model(model, pre_scaled)
    results[f"{name}_scaled"]     = [mse, rmse, mape]

metrics_df = pd.DataFrame(results, index=["MSE", "RMSE", "MAPE"]).T
print(metrics_df.to_string())
metrics_df


                                      MSE          RMSE      MAPE
LinearRegression_no_scaling  1.017415e+08  10086.700652  0.257000
LinearRegression_scaled      4.971825e+06   2229.758895  0.050503
Lasso_no_scaling             4.955739e+06   2226.148964  0.050414
Lasso_scaled                 4.955735e+06   2226.147921  0.050413
GradientBoosting_no_scaling  6.147298e+06   2479.374534  0.058211
GradientBoosting_scaled      6.122865e+06   2474.442309  0.057649


,MSE,RMSE,MAPE
LinearRegression_no_scaling,1.017415e+08,10086.700652,0.257000
LinearRegression_scaled,4.971825e+06,2229.758895,0.050503
Lasso_no_scaling,4.955739e+06,2226.148964,0.050414
Lasso_scaled,4.955735e+06,2226.147921,0.050413
GradientBoosting_no_scaling,6.147298e+06,2479.374534,0.058211
GradientBoosting_scaled,6.122865e+06,2474.442309,0.057649


### 1. Результаты моделей

Измененный датасет:

| Модель | MSE | RMSE | MAPE |
|--------|----------------|----------------|----------------|
| LinearRegression_no_scaling | 1.017415e+08 | 10086.70 | 0.2570 |
| LinearRegression_scaled | 4.971825e+06 | 2229.75 | 0.0505 |
| Lasso_no_scaling | 4.955739e+06 | 2226.14 | 0.0504 |
| Lasso_scaled | 4.955735e+06 | 2226.14 | 0.0504 |
| GradientBoosting_no_scaling | 6.147298e+06 | 2479.37 | 0.0582 |
| GradientBoosting_scaled | 6.122865e+06 | 2474.44 | 0.0576 |


### 1. Какая функция ошибки больше подходит и почему?

- Среди метрик MSE, RMSE и MAPE для данной задачи **лучше подходит RMSE**.  
- **Причина:**  
  - RMSE измеряет ошибку в тех же единицах, что и цена (рубли за квадратный метр), поэтому проще интерпретировать.  
  - MSE сильно масштабируется из-за возведения ошибки в квадрат и менее наглядна для оценки точности.  
  - MAPE показывает относительную ошибку, но чувствителен к маленьким значениям цены и может быть нестабильным, если цена близка к нулю.  
- На практике RMSE позволяет сразу видеть среднее отклонение предсказанной цены от фактической, что удобно для оценки качества модели.

---

### 2. Какие модели чувствительны к масштабу признаков, а какие почти инвариантны? Почему это важно для анализа признаков?

- **LinearRegression и Lasso**  
  - Обычно чувствительны к масштабу признаков, потому что величина коэффициентов напрямую зависит от единиц измерения признаков.  
  - Регуляризация привела данные к похожим порядкам, из-за чего нормализация не дала заметных изменений.

- **GradientBoosting**  
  - Практически инвариантен к масштабу признаков.  
  - Деревья строят разбиения по порогам, а не используют веса коэффициентов, поэтому масштаб числовых признаков не влияет на обучение.  

- **Вывод для анализа признаков:**  
  - Для линейных моделей нормализация критична для корректной интерпретации влияния каждого признака.  
  - Для деревьев и градиентного бустинга можно использовать исходные признаки без нормализации, что ускоряет обучение.